## Steering on natural images

This notebook loads concept vectors for 6 colors, then performs steering on a dataset of 60 images and plots results

In [4]:
model_name = 'qwen' #'qwen', 'intern' or 'gemma'

#### Import and utilities

In [1]:
import torch
import matplotlib.pyplot as plt, seaborn as sns
import pandas as pd
import hydra
from tqdm import tqdm
import numpy as np
import pickle, os
from transformers.utils import ModelOutput

import sys
sys.path.append('./src')

In [2]:
from transformers.utils import logging
logging.set_verbosity_error() 

In [ ]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
hydra.initialize("config")
cfg = hydra.compose("activations_gen", overrides=[dict(qwen='+experiment=bal_qwen7b',
                                                       intern='+experiment=bal_internvl8b',
                                                       gemma='+experiment=bal_gemma12b')[model_name],
                                                  ])
dataset_name = 'real_dataset'

/tmp/ipykernel_2579041/2126642052.py:2: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize("config")


In [7]:
colors=cfg.dataset.COLORS.copy()
shapes=cfg.dataset.SHAPES.copy()

In [ ]:
def _generate_proj_hook(layer_name : str,
                        prototypes: list = None):
  '''Generates an hook that performs 'projection patching', steering from prototypes[0]
  into prototypes[1].
  '''

  def proj_hook(model,input,output):
    if isinstance(output, tuple):
      #output = output[0] #usually other indexes are attentions
      tochange = output[0]['last_hidden_state']# without .cpu (even with .clone) raises error
    elif isinstance(output,torch.Tensor):
      tochange = output
    elif isinstance(output, ModelOutput):
      tochange = output['last_hidden_state'] # without .cpu (even with .clone) raises error
    # 'last_hidden_state' should be model-agnostic, but im not sure
    else:
      raise Exception(f"Hook in layer {layer_name}: output type unknown, {type(output)}")

    # print(tochange.shape)
    initshape = tochange.shape
    inittype = tochange.dtype
    tochangecloned = tochange.squeeze().to(torch.float)
    #print(initshape,tochangecloned.shape)
    if prototypes is None:
      raise Error
    del_prot = prototypes[0]
    ins_prot = prototypes[1]
    if len(tochangecloned.shape) == 3: #internvl splits one big image in subpictures, like 3×256×4k
      tochangecloned=tochangecloned.flatten(end_dim=1)
    
    for targettoken in range(tochangecloned.shape[0]): #make target appear
      target_proj_module = (tochangecloned[targettoken,:]@del_prot)
      tochangecloned[targettoken,:] = tochangecloned[targettoken,:]-target_proj_module*del_prot+target_proj_module*ins_prot
    
    tochange.copy_(tochangecloned.reshape(initshape).to(inittype))
    return output
  return proj_hook

In [8]:
metadf = pd.read_csv('data/real_dataset/metadata.csv',dtype=dict(ID=str))
metadata = metadf.to_dict('records')
metadf.head()

,ID,color,object,url,retrieved_date
0,000,red,pencil,https://cdn.pixabay.com/photo/2015/09/23/20/01...,2026_01_02
1,001,red,pencil,https://cdn.pixabay.com/photo/2020/02/11/09/42...,2026_01_02
2,002,red,notebook,https://cdn.pixabay.com/photo/2015/10/02/20/02...,2026_01_02
3,003,red,notebook,https://cdn.pixabay.com/photo/2016/08/17/13/16...,2026_01_02
4,004,red,flower,https://cdn.pixabay.com/photo/2017/07/06/11/51...,2026_01_02


In [ ]:
savedir = f'./outputs/{cfg.model.model_name}/{dataset_name}'
os.makedirs(savedir,exist_ok=True)
savedir

#### Load model

In [ ]:
model = hydra.utils.instantiate(cfg.model)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from transformers.utils import ModelOutput

In [ ]:
from transformers import GenerationConfig
if 'Intern' in cfg.model.model_name:
    generation_kwargs = dict(generation_config=GenerationConfig( 
    max_new_tokens= 8, 
    do_sample= False, #avoid sampling outputs (greedy choice)
    return_dict_in_generate= True,  #outputs is a dictionary, following args specify its contents
    output_attentions= False, # Passed to all submodels, needed to use hooks
    output_hidden_states= False,
    output_logits= True,  # These are handy
    output_scores= False,
    eos_token_id=92542
    ) )
    # I'll move this to the model class sooner or later
    def getstring(model,output,input_len):
      return model.processor.decode(output['sequences'][0])
else:
    generation_kwargs = { 
      "max_new_tokens": 8,
      "do_sample": False, #avoid sampling outputs (greedy choice)
      "return_dict_in_generate": True,  #outputs is a dictionary, following args specify its contents
      "output_attentions": False, # Passed to all submodels, needed to use hooks
      "output_hidden_states": False,
      "output_logits": True,  # These are handy
      "output_scores": False,
    }
    def getstring(model,output,input_len):
        return model.processor.decode(output['sequences'][0][input_len:])

#### Generate original (no steering) outputs

We let the model do whatever it wants with the resizing and tokenization of the image, but apply steering indiscriminately to all embedding tokens.

In [ ]:
probs = []
probs_ids = []
outs = []
logits = []
for meta in tqdm(metadata):
    img_path = f"./data/real_dataset/img/{meta['ID']}.png"
    prompt= f"Observe the image carefully. What color is the {meta['object']}? Answer in one word."
    model_input = model.get_inputs(img_path,prompt)
    with torch.inference_mode():
      generation_output = model.model.generate(**model_input,**generation_kwargs)
    input_len = model_input["input_ids"].shape[-1]
    str_out = getstring(model,generation_output,input_len)
    outputsm = torch.nn.functional.softmax(generation_output['logits'][0],dim=-1)
    top_probs, top_ids = (outputsm[0].topk(10)) #take top 10 tokens for the first word
    logits_out = (generation_output['logits'][0][0,top_ids].cpu())
    
    probs.append(top_probs.cpu())
    probs_ids.append(top_ids.cpu())
    outs.append(str_out) #full string
    logits.append(logits_out)

100%|██████████| 60/60 [03:37<00:00,  3.63s/it]


In [ ]:
generation_output['logits'][0].shape

torch.Size([1, 92553])

In [ ]:
# go from index to natural language
reverse_vocab = dict(zip(model.vocab.values(),model.vocab.keys()))

In [ ]:
top_tokens = []
for ids in probs_ids:
  topt=[]
  for id in ids:
    topt.append(reverse_vocab[id.item()])
  top_tokens.append(topt)

In [ ]:
i = 52
top_tokens[i],probs[i]

(['Blue', 'P', 'The', 'N', 'blue', 'purple', 'Dark', 'B', 'Light', 'V'],
 tensor([0.5497, 0.3428, 0.0818, 0.0063, 0.0052, 0.0035, 0.0030, 0.0009, 0.0007,
         0.0006]))

In [ ]:
color_is_correct = []
for i in range(60):
  answ = top_tokens[i][0]#.copy()
  color_is_correct.append(metadata[i]['color'] in outs[i].lower())
  # print(i,outs[i].replace("<|im_end|>",""),top_tokens[i][0],probs[i][0].item(),'\t',top_tokens[i][1],probs[i][1].item())

0  Red Red 0.8828058838844299 	 red 0.10702809691429138
1  Red Red 0.964987576007843 	 Orange 0.01369650848209858
2  Red Red 0.8746932148933411 	 The 0.05128996819257736
3  red red 0.7205416560173035 	 yellow 0.19406326115131378
4  Red Red 0.9750117063522339 	 The 0.019187357276678085
5  Red Red 0.5819683074951172 	 red 0.4102257192134857
6  Red Red 0.9616424441337585 	 The 0.0348222516477108
7  Red Red 0.9837136268615723 	 The 0.012118417769670486
8  Red Red 0.9676520824432373 	 The 0.022613130509853363
9  Red Red 0.9467614889144897 	 The 0.03964795172214508
10  Green Green 0.8729754686355591 	 The 0.11927375197410583
11  Green. Green 0.9395527839660645 	 The 0.05630701780319214
12  Green Green 0.7511950731277466 	 The 0.11934085935354233
13 The notebook is light The 0.3265596926212311 	 Light 0.23619219660758972
14  Green. Green 0.9534358382225037 	 The 0.02281259186565876
15  Green. Green 0.9146938323974609 	 Yellow 0.03548452630639076
16 Green and white. Green 0.9428991675376892 	 

In [ ]:
#List those images where the model got the color wrong
wrongslist = torch.arange(60)[~torch.tensor(color_is_correct)]
wrongslist

tensor([13, 52])

In [ ]:
resdict = {'probs':torch.stack(probs),
            'probs_ids':torch.stack(probs_ids),
            'outs':outs,
            'logits':torch.stack(logits),
            'ids':ids,
            'color_is_correct': torch.tensor(color_is_correct)
          }

In [ ]:
with open(os.path.join(savedir,'og_resdict.pkl'),'wb') as f:
  pickle.dump(resdict,f)

#### Generate steering outputs

5 model outputs for each image (one for each steering target color).

In [ ]:
template_label =  'centroid' 
## 'probe'
## 'probepca'
## 'centroid'

col_templates = torch.load(f'./outputs/{cfg.model.model_name}/color_cv_{template_label}.pt')[:6]

In [ ]:
prototypes = [] #this one will always have [conc vec to delete, conc vec to hallucinate]
handles  = model.register_hooks(hook_generator = _generate_proj_hook,
                        hook_layers = {'mmp': cfg.model.probe_layers['mmp']},
                        hook_generator_kwargs={
                                              'prototypes': prototypes,
                                              })
  ### handles['mmp'].remove() # to remove handles and run model without steering

In [ ]:

probs_st = []
probs_ids_st = []
outs_st = []
logits_st = []
colors_st = []
ids_st = []
with tqdm(total=len(colors)*len(metadata)) as pbar:
  for end_col in (colors[:]):
    for meta in (metadata[:]):
        if meta['color']!=end_col:
          prototypes.clear()
          prototypes.append(col_templates[coldict[meta['color']]].cuda())
          prototypes.append(col_templates[coldict[end_col]].cuda())

          img_path = f"./outputs/real_dataset/{meta['ID']}.png"
          prompt= f"Observe the image carefully. What color is the {meta['object']}? Answer in one word."
          model_input = model.get_inputs(img_path,prompt)
          with torch.inference_mode():
            generation_output = model.model.generate(**model_input,**generation_kwargs)
          input_len = model_input["input_ids"].shape[-1]
          str_out = getstring(model,generation_output,input_len)
          outputsm = torch.nn.functional.softmax(generation_output['logits'][0],dim=-1)
          top_probs, top_ids = (outputsm[0].topk(10)) #take top 10 tokens for the first word
          logits_out = (generation_output['logits'][0][0,top_ids].cpu())
          
          probs_st.append(top_probs.cpu())
          probs_ids_st.append(top_ids.cpu())
          outs_st.append(str_out) #full string
          logits_st.append(logits_out)
          colors_st.append([meta['color'],end_col])
          ids_st.append(meta['ID'])
        pbar.update()

In [ ]:
# go from index to natural language
reverse_vocab = dict(zip(model.vocab.values(),model.vocab.keys()))

In [ ]:
top_tokens_st = []
for ids in probs_ids_st:
  topt = []
  for id in ids:
    topt.append(reverse_vocab[id.item()])
  top_tokens_st.append(topt)

In [ ]:
color_is_correct_st = []
for i in range(len(outs_st)):
  answ = outs_st[i]
  color_is_correct_st.append(colors_st[i][1] in answ.lower())
  # print(i,top_tokens_st[i][0],probs_st[i][0].item(),'\t',top_tokens_st[i][1],probs_st[i][1].item())

In [ ]:
color_is_oldone = []
for i in range(len(outs_st)):
  answ = outs_st[i]
  color_is_oldone.append(colors_st[i][0] in answ.lower())

In [ ]:
np.sum(color_is_oldone)

In [ ]:
start_colors=np.array(colors_st)[:,0]
end_colors=np.array(colors_st)[:,1]

resdict_st = {'probs_st':torch.stack(probs_st),
            'probs_ids_st':torch.stack(probs_ids_st),
            'outs_st':outs_st,
            'logits_st':torch.stack(logits_st),
            'colors_st': colors_st,
            'ids_st':ids_st,
            'color_is_correct_st': torch.tensor(color_is_correct_st)
          }

In [ ]:
steer_df = pd.DataFrame(dict(image_id = ids_st,
                             start = start_colors,end=end_colors,
                             is_correct=color_is_correct_st,
                             #out = outs[10:60],
                             steer_out=outs_st,
                             out0=[i[0] for i in top_tokens_st],
                             P0 = [p[0].item() for p in probs_st],
                             out1=[i[1] for i in top_tokens_st],
                             P1 = [p[1].item() for p in probs_st],
                             out2=[i[2] for i in top_tokens_st],
                             P2 = [p[2].item() for p in probs_st],
                             out3=[i[3] for i in top_tokens_st],
                             P3 = [p[3].item() for p in probs_st],
                             )
)

In [ ]:
steer_df.to_csv(os.path.join(savedir,template_label+'_steerdf_.csv'),index=False)
with open(os.path.join(savedir,template_label+'_steerresdict_.pkl'),'wb') as f:
  pickle.dump(resdict_st,f)

In [ ]:
steer_df['is_correct'].value_counts(normalize=True)

In [ ]:
# exclude images for which the model was wrong to begin with 
if len(wrongslist)==0:
  print('No image excluded')
  accs = steer_df['is_correct'].value_counts(normalize=True)
  nsteers = len(steer_df)
else :
  print("excluding image ids", wrongslist)
  dfcopy = steer_df.copy()
  for i in wrongslist:
    dfcopy.drop(dfcopy[dfcopy['image_id']==i].index, inplace=True)
  accs = dfcopy['is_correct'].value_counts(normalize=True)
  nsteers = len(dfcopy)
accs, nsteers

### Print pre-computed results

In [59]:
print("\t\t\tprobe   pca  centroid")
for mod in ["Qwen2_5_7B","InternVL2_5_8B","Gemma3_12B"]:
  sdir = f'./outputs/{mod}/{dataset_name}'
  # print(sdir)
  with open(os.path.join(sdir,'og_resdict.pkl'),'rb') as f:
    res_clean = pickle.load(f)
  df_probe = pd.read_csv(os.path.join(sdir,'probe_steerdf.csv'))
  df_pca = pd.read_csv(os.path.join(sdir,'probepca_steerdf.csv'))
  df_cent = pd.read_csv(os.path.join(sdir,'centroid_steerdf.csv'))
  og_colcorrect = res_clean['color_is_correct']
  wrongslist = torch.arange(60)[~(og_colcorrect)].tolist()
  
  df_probe = df_probe[~df_probe['image_id'].isin(wrongslist)]
  df_pca = df_pca[~df_pca['image_id'].isin(wrongslist)]
  df_cent = df_cent[~df_cent['image_id'].isin(wrongslist)]

  probe_acc = df_probe['is_correct'].values.mean()
  pca_acc = df_pca['is_correct'].values.mean()
  cent_acc = df_cent['is_correct'].values.mean()
  print(mod,f"({df_cent.shape[0]})",f"\t{probe_acc*100:5.2f}% {pca_acc*100:5.2f}% {cent_acc*100:5.2f}%")

			probe   pca  centroid
Qwen2_5_7B (300) 	 3.67% 32.33% 85.00%
InternVL2_5_8B (290) 	62.76% 93.45% 90.34%
Gemma3_12B (300) 	 3.67% 16.33% 96.00%
